# Session 3: MongoDB Operators

Diploma in AI · companion notebook

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom. In Colab, the first cell installs anything that is missing.

In [1]:
import os, sys, subprocess
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mongomock', 'pymongo<4.9'], check=False)
    # fetch helper/data files used by the examples
    base = 'https://raw.githubusercontent.com/Hamnakhan123/Diploma-in-AI/main/term-1/02-managing-large-datasets-with-mongodb/code/'
    for f in ['store_db.py']:
        if not os.path.exists(f):
            subprocess.run(['wget', '-q', base + f], check=False)
print('ready')

ready


## Example: `session-03_query_operators.py`

In [2]:
# Session 3 · Comparison, logical, element and evaluation operators
from store_db import get_db, show

db = get_db()
P = {"_id": 0, "name": 1, "price": 1, "stock": 1}

print("1) $gt / $lte: price above 15,000 and at most 60,000")
show(db.products.find({"price": {"$gt": 15000, "$lte": 60000}}, P))

print("\n2) $in / $nin: categories Audio or Storage")
show(db.products.find({"category": {"$in": ["Audio", "Storage"]}}, P))

print("\n3) $ne: products that are not out of stock (count)")
print(db.products.count_documents({"stock": {"$ne": 0}}))

print("\n4) $or: cheap (< 2,000) OR nearly sold out (stock <= 3)")
show(db.products.find({"$or": [{"price": {"$lt": 2000}}, {"stock": {"$lte": 3}}]}, P))

print("\n5) $and on the same field (two conditions on price)")
show(db.products.find({"$and": [{"price": {"$gte": 4000}}, {"price": {"$lte": 7000}}]}, P))

print("\n6) $nor: neither Accessories nor priced over 100,000")
show(db.products.find({"$nor": [{"category": "Accessories"}, {"price": {"$gt": 100000}}]}, P))

print("\n7) $not: name does NOT start with a capital M")
print([d["name"] for d in db.products.find({"name": {"$not": {"$regex": "^M"}}})])

print("\n8) $exists vs null: customers with no usable email")
print("field missing   :", [c["name"] for c in db.customers.find({"email": {"$exists": False}})])
print("missing OR null :", [c["name"] for c in db.customers.find({"email": None})])

print("\n9) $type: documents where price is stored as an int")
print(db.products.count_documents({"price": {"$type": "int"}}), "of", db.products.count_documents({}))

print("\n10) $regex (case-insensitive) on name")
print([d["name"] for d in db.products.find({"name": {"$regex": "phone", "$options": "i"}})])

print("\n11) $expr compares two fields of the same document: stock value over 200,000")
show(db.products.find({"$expr": {"$gt": [{"$multiply": ["$price", "$stock"]}, 200000]}}, P))

print("\n12) $mod inside $expr: even product ids")
print([d["_id"] for d in db.products.find({"$expr": {"$eq": [{"$mod": ["$_id", 2]}, 0]}})])

1) $gt / $lte: price above 15,000 and at most 60,000
{"name": "27-inch Monitor", "price": 52000, "stock": 6}
{"name": "Noise-Cancelling Headphones", "price": 18500, "stock": 9}
{"name": "Portable SSD 1TB", "price": 16000, "stock": 18}
(3 documents)

2) $in / $nin: categories Audio or Storage
{"name": "Noise-Cancelling Headphones", "price": 18500, "stock": 9}
{"name": "Bluetooth Speaker", "price": 6500, "stock": 22}
{"name": "Portable SSD 1TB", "price": 16000, "stock": 18}
(3 documents)

3) $ne: products that are not out of stock (count)
9

4) $or: cheap (< 2,000) OR nearly sold out (stock <= 3)
{"name": "Wireless Mouse", "price": 1500, "stock": 40}
{"name": "USB-C Hub", "price": 4200, "stock": 0}
{"name": "Gaming Laptop", "price": 285000, "stock": 3}
(3 documents)

5) $and on the same field (two conditions on price)
{"name": "USB-C Hub", "price": 4200, "stock": 0}
{"name": "Bluetooth Speaker", "price": 6500, "stock": 22}
{"name": "Webcam HD", "price": 4500, "stock": 12}
(3 documents)



## Example: `session-03_array_operators.py`

In [3]:
# Session 3 · Array query, update and projection operators
from store_db import get_db, show

db = get_db()

print("1) $all: tagged BOTH wireless and office")
print([d["name"] for d in db.products.find({"tags": {"$all": ["wireless", "office"]}})])

print("\n2) $size: exactly one tag")
print([d["name"] for d in db.products.find({"tags": {"$size": 1}})])

print("\n3) $elemMatch: orders with ONE item that has qty >= 2 AND price >= 4000")
print([o["_id"] for o in db.orders.find({"items": {"$elemMatch": {"qty": {"$gte": 2}, "price": {"$gte": 4000}}}})])
print("   without $elemMatch the two conditions may match different items:")
print([o["_id"] for o in db.orders.find({"items.qty": {"$gte": 2}, "items.price": {"$gte": 4000}})])

print("\n4) $push with $each and $slice: keep only the 3 newest tags")
db.products.update_one({"_id": 101}, {"$push": {"tags": {"$each": ["ergonomic", "sale"], "$slice": -3}}})
print(db.products.find_one({"_id": 101})["tags"])

print("\n5) $addToSet adds only if missing")
db.products.update_one({"_id": 101}, {"$addToSet": {"tags": {"$each": ["sale", "bestseller"]}}})
print(db.products.find_one({"_id": 101})["tags"])

print("\n6) $pull removes matching elements; $pop removes first (-1) or last (1)")
db.products.update_one({"_id": 101}, {"$pull": {"tags": "sale"}})
db.products.update_one({"_id": 101}, {"$pop": {"tags": -1}})
print(db.products.find_one({"_id": 101})["tags"])

print("\n7) positional $: change the price of product 101 inside order 5001 only")
db.orders.update_one({"_id": 5001, "items.productId": 101}, {"$set": {"items.$.price": 1400}})
show(db.orders.find({"_id": 5001}, {"items": 1}))

print("\n8) projection $slice: first item of each delivered order")
show(db.orders.find({"status": "Delivered"}, {"items": {"$slice": 1}, "customerId": 1}).limit(3))

1) $all: tagged BOTH wireless and office
['Wireless Mouse']

2) $size: exactly one tag
['Gaming Laptop']

3) $elemMatch: orders with ONE item that has qty >= 2 AND price >= 4000
[5006, 5007, 5009]
   without $elemMatch the two conditions may match different items:
[5001, 5006, 5007, 5009]

4) $push with $each and $slice: keep only the 3 newest tags
['office', 'ergonomic', 'sale']

5) $addToSet adds only if missing
['office', 'ergonomic', 'sale', 'bestseller']

6) $pull removes matching elements; $pop removes first (-1) or last (1)
['ergonomic', 'bestseller']

7) positional $: change the price of product 101 inside order 5001 only
{"items": [{"productId": 101, "qty": 2, "price": 1400}, {"productId": 107, "qty": 1, "price": 4500}], "_id": 5001}
(1 document)

8) projection $slice: first item of each delivered order
{"customerId": 1, "_id": 5001, "items": [{"productId": 101, "qty": 2, "price": 1400}]}
{"customerId": 2, "_id": 5002, "items": [{"productId": 103, "qty": 1, "price": 52000}]}
{